# 7. Inference Pipeline

Applies the trained PointNet++ classifier to new LiDAR tiles.

**Steps:**
1. Load trained model and class map
2. For each tile: check if BGT-labeled LAZ exists, otherwise run preprocessing
3. Extract DBSCAN obstacle clusters via `cluster_io`
4. Run model forward pass + fallback rules
5. Export GeoJSON with predicted class, confidence, and source per cluster
6. Tile overview visualization

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('.').resolve()))

from config import LABELED_DIR, CLUSTERS_DIR, MODELS_DIR, DATASET_DIR

# ── Config ──────────────────────────────────────────────────────────────────
# Tiles to run inference on (can be different from training set)
INFERENCE_TILECODES = [
    # Add tilecodes here, e.g. "120300_488900"
]

CONFIDENCE_THRESH   = 0.60  # below this: fall back to RF
N_POINTS            = 512
N_FEATURES          = 7
OUTPUT_DIR          = CLUSTERS_DIR / "inference"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load model and class map

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import joblib
from utils.pointnet2 import build_model
from utils.labels import Labels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load class map
with open(DATASET_DIR / "class_map.json") as f:
    label_map = {int(k): int(v) for k, v in json.load(f).items()}

n_classes = len(set(label_map.values()))

# Reverse map: class_idx → dominant label code
idx_to_code = {}
for code, idx in label_map.items():
    if idx not in idx_to_code:
        idx_to_code[idx] = code

# Label names
custom_labels = {}
custom_path = CLUSTERS_DIR / "custom_labels.json"
if custom_path.exists():
    with open(custom_path) as f:
        custom_labels = {int(k): v for k, v in json.load(f).items()}
label_names = dict(Labels.STR_DICT)
label_names.update(custom_labels)

# Load PointNet++ model
model = build_model(n_classes=n_classes, n_features=N_FEATURES, device=str(device))
model.load_state_dict(torch.load(MODELS_DIR / "pointnet2_classifier.pt", map_location=device))
model.eval()
print(f"PointNet++ loaded  |  {n_classes} classes  |  device: {device}")

# Load RF fallback
rf_fallback = None
rf_path = MODELS_DIR / "rf_baseline.pkl"
if rf_path.exists():
    rf_fallback = joblib.load(rf_path)
    print(f"RF fallback loaded")

## Predict single cluster helper

In [ ]:
from utils.cluster_dataset import furthest_point_sample

def predict_cluster(npz_path):
    """
    Returns (label_code, label_name, confidence, source).
    source: 'bgt_prior' | 'pn2' | 'rf_fallback' | 'rule_min_pts'
    """
    data  = np.load(npz_path, allow_pickle=False)
    xyz_c = data['xyz_centered'].astype(np.float32)
    rgb   = data['rgb_norm'].astype(np.float32)
    h_ag  = data['height_ag'].astype(np.float32)
    n_pts = int(data['n_raw_pts'])

    # Rule: too few points for PointNet++
    if n_pts < 30:
        code = int(data['label']) if int(data['label']) > 0 else 99
        return code, label_names.get(code, str(code)), 1.0, 'rule_min_pts'

    # BGT prior with high confidence — keep it
    auto_label = int(data['label'])
    label_frac = float(data['label_frac'])
    if auto_label > 0 and label_frac >= 0.80:
        return auto_label, label_names.get(auto_label, str(auto_label)), label_frac, 'bgt_prior'

    # Build (N, 7) feature matrix
    pts7 = np.column_stack([xyz_c, rgb, h_ag]).astype(np.float32)
    N    = len(pts7)
    if N >= N_POINTS:
        idx = furthest_point_sample(pts7[:, :3], N_POINTS)
    else:
        idx = np.concatenate([np.arange(N),
                               np.random.choice(N, N_POINTS - N, replace=True)])
    pts7 = pts7[idx]

    # PointNet++ inference
    with torch.no_grad():
        t = torch.from_numpy(pts7).unsqueeze(0).to(device)  # (1, N, 7)
        probs = torch.softmax(model(t), dim=1).squeeze(0).cpu().numpy()

    pred_idx = int(probs.argmax())
    conf     = float(probs[pred_idx])
    code     = idx_to_code.get(pred_idx, 99)

    if conf >= CONFIDENCE_THRESH:
        return code, label_names.get(code, str(code)), conf, 'pn2'

    # RF fallback
    if rf_fallback is not None:
        dx = float(xyz_c[:, 0].max() - xyz_c[:, 0].min())
        dy = float(xyz_c[:, 1].max() - xyz_c[:, 1].min())
        width = min(dx, dy); length = max(dx, dy)
        height = float(h_ag.max() - h_ag.min())
        features = np.array([[dx, dy, width, length, height,
                               length / max(width, 0.01),
                               float(data.get('area_m2', 0.0)),
                               float(N), float(h_ag.mean()), float(h_ag.std()), float(h_ag.max()),
                               float(rgb[:, 0].mean()), float(rgb[:, 1].mean()), float(rgb[:, 2].mean()),
                               float(rgb[:, 1].mean() - rgb[:, 0].mean())]])
        rf_pred_idx = int(rf_fallback.predict(features)[0])
        rf_code     = idx_to_code.get(rf_pred_idx, 99)
        return rf_code, label_names.get(rf_code, str(rf_code)), conf, 'rf_fallback'

    return code, label_names.get(code, str(code)), conf, 'pn2_low_conf'

## Run inference on tiles

In [ ]:
import traceback
from utils.cluster_io import extract_and_save_clusters, build_cluster_inventory

if not INFERENCE_TILECODES:
    print("No tilecodes specified in INFERENCE_TILECODES. Add tilecodes in the config cell above.")
else:
    for tilecode in INFERENCE_TILECODES:
        print(f"\nProcessing {tilecode} …")
        bgt_laz = LABELED_DIR / f"bgt_labeled_{tilecode}.laz"
        if not bgt_laz.exists():
            print(f"  ⚠  No BGT-labeled file found — run notebook 4 first.")
            continue

        # Extract clusters
        tile_cluster_dir = CLUSTERS_DIR / "inference_clusters"
        tile_cluster_dir.mkdir(exist_ok=True)
        inv = extract_and_save_clusters(bgt_laz, tile_cluster_dir, tilecode)
        print(f"  Clusters extracted: {len(inv)}")

        # Predict each cluster
        results = []
        for _, row in inv.iterrows():
            try:
                code, name, conf, src = predict_cluster(row['npz_path'])
            except Exception as e:
                code, name, conf, src = 99, 'Noise', 0.0, 'error'
            results.append({
                'tilecode':     tilecode,
                'cluster_idx':  row['cluster_idx'],
                'centroid_x':   row['centroid_x'],
                'centroid_y':   row['centroid_y'],
                'area_m2':      row['area_m2'],
                'n_raw_pts':    row['n_raw_pts'],
                'pred_label':   code,
                'pred_name':    name,
                'confidence':   round(conf, 3),
                'source':       src,
            })

        results_df = pd.DataFrame(results)
        print(f"  Predictions:")
        print(results_df['pred_name'].value_counts().to_string())

        # Export GeoJSON
        import geopandas as gpd
        from shapely.geometry import Point
        gdf = gpd.GeoDataFrame(
            results_df,
            geometry=[Point(r['centroid_x'], r['centroid_y']) for _, r in results_df.iterrows()],
            crs='EPSG:28992',
        )
        out_path = OUTPUT_DIR / f"predictions_{tilecode}.geojson"
        gdf.to_file(str(out_path), driver='GeoJSON')
        print(f"  Saved: {out_path}")

## Tile overview visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

LABEL_COLORS = {
    30: '#2ca02c', 40: '#d62728', 49: '#ff9896', 60: '#ff7f0e',
    80: '#9467bd', 81: '#8c564b', 83: '#e377c2', 44: '#aec7e8',
    46: '#1f77b4', 47: '#17becf', 45: '#bcbd22', 65: '#7f7f7f', 99: '#cccccc',
}

for tilecode in INFERENCE_TILECODES:
    geojson_path = OUTPUT_DIR / f"predictions_{tilecode}.geojson"
    if not geojson_path.exists():
        continue

    import geopandas as gpd
    gdf = gpd.read_file(str(geojson_path))

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.set_facecolor('#1a1a1a'); fig.patch.set_facecolor('#1a1a1a')

    handles = {}
    for _, row in gdf.iterrows():
        code = int(row['pred_label'])
        col  = LABEL_COLORS.get(code, '#888888')
        size = max(15, min(80, int(row['confidence'] * 80)))
        ax.scatter(row.geometry.x, row.geometry.y, c=col, s=size, linewidths=0, zorder=3)
        if code not in handles:
            handles[code] = mpatches.Patch(color=col, label=f"{row['pred_name']} ({(gdf['pred_label']==code).sum()})")

    ax.legend(handles=list(handles.values()), fontsize=7, loc='upper right',
              facecolor='#2a2a2a', edgecolor='#555', labelcolor='white')
    ax.set_aspect('equal')
    ax.set_title(f'{tilecode} — {len(gdf)} clusters', color='white', fontsize=10)
    ax.tick_params(colors='grey')
    for sp in ax.spines.values(): sp.set_edgecolor('#444')

    plt.tight_layout()
    out_img = OUTPUT_DIR / f"overview_{tilecode}.png"
    plt.savefig(str(out_img), dpi=120, facecolor='#1a1a1a')
    plt.show()
    print(f"Saved: {out_img}")